# Huấn luyện YOLO cho camera xe thật
YOLO (mạng phát hiện vật thể trong một lượt suy luận) lấy tên và thứ tự class (lớp đối tượng) trực tiếp từ dataset (bộ dữ liệu). Mặc định YOLOv8n; có thể chọn YOLO11n, cần kiểm tra tương thích TensorRT 8.2 trên xe.

Chạy từ trên xuống. `FAKE_DATA=True` chạy một epoch (lượt đi hết dữ liệu) trên CPU với ảnh 64×64 và trọng số khởi tạo ngẫu nhiên, không tải trọng số COCO (bộ dữ liệu tiền huấn luyện). Chế độ thật mặc định 320×320 và học chuyển giao. Các kết quả giả chỉ kiểm tra quy trình.

Tăng cường camera được tạo **sau khi chia vị trí**, chỉ trong tập huấn luyện. Không lật ngang vì biển có hướng. Kết quả nằm trong `MyDrive/UIT_Car_Racing/Vong3/`.


In [ ]:
TASK = "yolo"
PROJECT = "ten-project-vat-the"
WORKSPACE = None
VERSION = 1
MODEL = "yolov8n"  # Hoặc "yolo11n".
IMGSZ = 320
EPOCHS = 100
BATCH = 16
SEED = 42
FAKE_DATA = True
FAKE_IMGSZ = 64
INSTALL_DEPS = True
LOCAL_ROOT = "./MyDrive/UIT_Car_Racing/Vong3"
VAL_RATIO = 0.2
VAL_GROUPS = []  # Ví dụ ["target_3"], giữ cùng mục tiêu ở mọi phiên/lượt.
AUG_COPIES = 2  # Số biến thể camera cho mỗi ảnh train; giữ cả ảnh gốc.
CONF_THRESHOLDS = [0.1,0.2,0.3,0.4,0.5,0.6,0.7]
EVAL_CONF = 0.25
MATCH_IOU = 0.5
NMS_IOU = 0.45  # NMS: loại khung trùng theo từng lớp.
BRIGHTNESS = [0.6,0.8,1.0,1.2,1.4]
MODE = "train"  # "train" hoặc "prelabel" (gán nhãn sơ bộ).
PRELABEL_WEIGHTS = ""  # Đường dẫn .pt đã huấn luyện, bên trong Vong3.
PRELABEL_CONF = 0.35
PRELABEL_LIMIT = 20  # None nếu muốn xử lý hết ảnh đang chờ gán nhãn.


In [ ]:
import os, sys, subprocess, importlib.util
from pathlib import Path
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
if INSTALL_DEPS:
    packages = ["torch", "torchvision", "numpy<3", "opencv-python-headless", "PyYAML", "onnx>=1.14,<2", "onnxruntime>=1.16,<2", "thop", "matplotlib", "roboflow", "requests"]
    if TASK == "yolo":
        packages += ["ultralytics==8.4.161", "onnxslim"]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/UIT_Car_Racing/Vong3")
else:
    ROOT = Path(LOCAL_ROOT).expanduser().resolve()
ROOT.mkdir(parents=True, exist_ok=True)
# Mọi dữ liệu, bộ nhớ đệm và kết quả đều ở Vong3.
os.environ["YOLO_CONFIG_DIR"] = str(ROOT / "cache" / "ultralytics")
os.environ["TORCH_HOME"] = str(ROOT / "cache" / "torch")
os.environ["MPLCONFIGDIR"] = str(ROOT / "cache" / "matplotlib")
import re, csv, json, random, shutil, hashlib, datetime, getpass
import numpy as np
import cv2, yaml, torch
import torch.nn as nn
import torch.nn.functional as F
import onnx, onnxruntime as ort
from thop import profile
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(2 if FAKE_DATA else max(1, min(8, os.cpu_count() or 1)))
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = "cpu" if FAKE_DATA else ("cuda:0" if torch.cuda.is_available() else "cpu")
RUN_ID = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")
RUN = ROOT / TASK / ("fake" if FAKE_DATA else "real") / RUN_ID
RUN.mkdir(parents=True, exist_ok=False)
print("Thiết bị:", DEVICE, "| Kết quả:", RUN)

def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")

def write_csv(path, records, fields):
    with Path(path).open("w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields); w.writeheader(); w.writerows(records)

def rf_project():
    from roboflow import Roboflow
    key = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            key = userdata.get("ROBOFLOW_API_KEY")
        except Exception:
            pass
    if not key:
        key = getpass.getpass("Khoá Roboflow (không lưu): ")
    rf = Roboflow(api_key=key)
    del key
    return rf.workspace(WORKSPACE).project(PROJECT) if WORKSPACE else rf.workspace().project(PROJECT)

RF_SUFFIX = re.compile(r"_(?:jpg|jpeg|png|bmp)\.rf\.[A-Za-z0-9]+$", re.I)
NAME_RE = re.compile(r"^(?P<session>.+?)_p(?P<pass_id>\d+)_l(?P<luot>\d+)_t(?P<target>\d+)_(?P<frame>\d+)$")

def original_stem(path):
    return RF_SUFFIX.sub("", Path(path).stem)

def metadata(path):
    stem = original_stem(path)
    m = NAME_RE.fullmatch(stem)
    if not m:
        raise ValueError("Tên ảnh không có lượt/mục tiêu/pass: " + stem)
    d = m.groupdict()
    for k in ("pass_id", "luot", "target", "frame"):
        d[k] = int(d[k])
    d["stem"] = stem
    # Cùng mục tiêu ở các lượt/phiên phải ở cùng tập. Với t00 (đường liên tục),
    # chưa có vị trí cụ thể: giữ nguyên cả lượt, không chia từng khung hay pass.
    d["group"] = "target_{}".format(d["target"]) if d["target"] else "road_l{}".format(d["luot"])
    return d

def split_records(records):
    groups = sorted(set(r["group"] for r in records))
    if len(groups) < 2:
        raise ValueError("Cần ít nhất 2 vị trí; t00 cần ít nhất 2 lượt độc lập.")
    if VAL_GROUPS:
        vg = set(VAL_GROUPS)
        if not vg <= set(groups):
            raise ValueError("VAL_GROUPS chứa vị trí không tồn tại")
    else:
        shuffled = groups.copy(); random.Random(SEED).shuffle(shuffled)
        vg = set(shuffled[:max(1, min(len(groups)-1, round(len(groups)*VAL_RATIO)))])
    for r in records:
        r["split"] = "val" if r["group"] in vg else "train"
    train = [r for r in records if r["split"] == "train"]
    val = [r for r in records if r["split"] == "val"]
    assert train and val
    assert not set(r["group"] for r in train) & set(r["group"] for r in val)
    write_csv(RUN / "split.csv", [{k:r[k] for k in ("stem","session","luot","target","pass_id","group","split")} for r in records],
              ["stem","session","luot","target","pass_id","group","split"])
    print("Train:", len(train), "| Val:", len(val), "| Vị trí val:", sorted(vg))
    return train, val


## Lấy dữ liệu và chia vị trí
Roboflow xuất dạng `yolov8`; gộp train/valid/test rồi chia lại theo mục tiêu. Với t00, giữ cả lượt thu ở cùng tập. `VAL_GROUPS` cho phép chủ động giữ các vị trí cụ thể. Không tạo phiên bản Roboflow có tăng cường dữ liệu vì các bản sao có thể rò sang val (tập đánh giá).

Ảnh phải theo mẫu `phiên_p<pass>_l<lượt>_t<mục tiêu>_<khung>.jpg`. Thiếu file nhãn sẽ dừng; ảnh nền phải có file `.txt` rỗng. Nhãn phát hiện gồm đúng năm số: lớp, tâm x/y, rộng/cao, toạ độ chuẩn hoá 0–1. Không dùng nhãn đa giác trong notebook này.


In [ ]:
from ultralytics import YOLO
assert MODEL in ("yolov8n","yolo11n") and MODE in ("train","prelabel")
SIZE = FAKE_IMGSZ if FAKE_DATA else IMGSZ
assert isinstance(SIZE,int) and SIZE>=64 and SIZE%32==0
DATA=RUN/"dataset"; DATA.mkdir()
if FAKE_DATA:
    folder=DATA/"train"
    (folder/"images").mkdir(parents=True); (folder/"labels").mkdir()
    (DATA/"data.yaml").write_text(yaml.safe_dump({"names":["bien_thu","vat_thu"],"nc":2}),encoding="utf-8")
    rng=np.random.default_rng(SEED)
    for i in range(24):
        stem="fake_p{:03d}_l{}_t{:02d}_{:04d}".format(i//6+1,i%4+1,i%4+1,i)
        im=np.full((SIZE,SIZE,3),60,np.uint8)
        c=i%2; cv2.rectangle(im,(SIZE//4,SIZE//4),(3*SIZE//4,3*SIZE//4),(30,180,220) if c else (200,80,60),-1)
        im=np.clip(im.astype(float)+rng.normal(0,2,im.shape),0,255).astype(np.uint8)
        cv2.imwrite(str(folder/"images"/(stem+".jpg")),im)
        (folder/"labels"/(stem+".txt")).write_text("{} 0.5 0.5 0.5 0.5\n".format(c),encoding="utf-8")
else:
    project=rf_project()
    project.version(VERSION).download("yolov8",location=str(DATA),overwrite=True)

spec=yaml.safe_load((DATA/"data.yaml").read_text(encoding="utf-8"))
names=spec["names"]
if isinstance(names,dict):
    mapping={int(k):v for k,v in names.items()}
    assert sorted(mapping)==list(range(len(mapping))), "Số hiệu lớp phải liên tiếp từ 0"
    names=[mapping[i] for i in range(len(mapping))]
CLASS_NAMES=list(names); NC=len(CLASS_NAMES)
assert NC and len(set(CLASS_NAMES))==NC and all(isinstance(n,str) for n in CLASS_NAMES)
assert int(spec.get("nc",NC))==NC
records=[]; seen=set(); GT={}
for p in sorted(DATA.glob("*/images/*")):
    if p.suffix.lower() not in (".jpg",".jpeg",".png",".bmp"): continue
    r=metadata(p)
    if r["stem"] in seen: raise ValueError("Ảnh trùng tên gốc: "+r["stem"])
    seen.add(r["stem"])
    im=cv2.imread(str(p))
    if im is None: raise ValueError("Ảnh không đọc được: "+p.name)
    label=p.parent.parent/"labels"/(p.stem+".txt")
    if not label.exists(): raise ValueError("Thiếu file nhãn: "+label.name)
    rows=[]
    for line in label.read_text(encoding="utf-8").splitlines():
        if not line.strip(): continue
        v=np.array([float(x) for x in line.split()])
        if len(v)!=5 or not np.isfinite(v).all(): raise ValueError("Nhãn phải gồm 5 số hữu hạn: "+label.name)
        c,x,y,w,h=v
        if c!=int(c) or not 0<=c<NC or min(w,h)<=0 or min(x-w/2,y-h/2)<-1e-5 or max(x+w/2,y+h/2)>1+1e-5:
            raise ValueError("Nhãn ngoài miền hợp lệ: "+label.name)
        rows.append(v)
    r.update(image=p,label=label); records.append(r)
    GT[r["stem"]]=np.array(rows,dtype=np.float32).reshape(-1,5)
train,val=split_records(records)
for c,n in enumerate(CLASS_NAMES):
    count=[sum(int((GT[r["stem"]][:,0]==c).sum()) for r in part) for part in (train,val)]
    print(n,"| Số vật train/val:",count)
    if min(count)==0: print("Cảnh báo: lớp thiếu mẫu ở một tập; cần xem lại vị trí hoặc bổ sung dữ liệu.")


In [ ]:
def camera_aug(image, rng):
    """Chỉ đổi màu/cường độ, giữ nguyên hình học để khung và mask còn đúng."""
    x = image.astype(np.float32)
    mean = x.mean(axis=(0,1), keepdims=True)
    x = (x-mean)*rng.uniform(0.8,1.2)+mean
    x *= rng.uniform(0.75,1.25)
    x *= rng.uniform(0.85,1.15,(1,1,3))  # Lệch cân bằng trắng.
    h,w = x.shape[:2]
    if rng.random() < 0.4:
        # Một vùng hầm tối và ám màu đèn, không phải làm tối toàn ảnh.
        x0 = int(rng.uniform(0,0.35)*w); x1 = int(rng.uniform(0.65,1)*w)
        y0 = int(rng.uniform(0,0.45)*h)
        tint = np.array([rng.uniform(0.6,1.0),rng.uniform(0.65,1.0),rng.uniform(0.85,1.2)])
        x[y0:,x0:x1] *= rng.uniform(0.12,0.4)*tint
    if rng.random() < 0.3:
        k = int(rng.choice([3,5,7])); kernel = np.zeros((k,k),np.float32)
        if rng.random() < 0.5:
            kernel[k//2,:] = 1/k
        else:
            kernel[:,k//2] = 1/k
        x = cv2.filter2D(x,-1,kernel)
    if rng.random() < 0.5:
        x += rng.normal(0,rng.uniform(1,9),x.shape)
    if rng.random() < 0.3:
        yy,xx = np.mgrid[:h,:w]
        stripe = xx - rng.uniform(0.1,0.9)*w - rng.uniform(-0.4,0.4)*yy
        glare = np.exp(-0.5*(stripe/max(1,w*rng.uniform(0.015,0.06)))**2)
        x += glare[...,None]*rng.uniform(50,160)
    return np.clip(x,0,255).astype(np.uint8)

# Lưu ảnh xem trước để đối chiếu tăng cường dữ liệu với ảnh gốc.
def preview_aug(image):
    rng = np.random.default_rng(SEED)
    tiles = [image]+[camera_aug(image,rng) for _ in range(5)]
    cv2.imwrite(str(RUN / "augmentation.jpg"), np.concatenate(tiles,axis=1))


In [ ]:
SPLIT=RUN/"split_dataset"
for split in ("train","val"):
    for sub in ("images","labels"): (SPLIT/split/sub).mkdir(parents=True)
rng=np.random.default_rng(SEED)
manifest=[]
for r in records:
    dest=SPLIT/r["split"]; stem=r["stem"]
    shutil.copy2(r["image"],dest/"images"/(stem+r["image"].suffix))
    shutil.copy2(r["label"],dest/"labels"/(stem+".txt"))
    if r["split"]=="train":
        for j in range(1 if FAKE_DATA else AUG_COPIES):
            name=stem+"_aug{:02d}".format(j)
            image=camera_aug(cv2.imread(str(r["image"])),rng)
            cv2.imwrite(str(dest/"images"/(name+".jpg")),image)
            shutil.copy2(r["label"],dest/"labels"/(name+".txt"))
            manifest.append(dict(augmented=name,source=stem,group=r["group"],split="train"))
write_csv(RUN/"augmentation_manifest.csv",manifest,["augmented","source","group","split"])
assert all(r["group"] not in set(v["group"] for v in val) for r in manifest)
preview_aug(cv2.imread(str(train[0]["image"])))
DATA_YAML=RUN/"data.yaml"
DATA_YAML.write_text(yaml.safe_dump({"path":str(SPLIT.resolve()),"train":"train/images","val":"val/images","nc":NC,"names":CLASS_NAMES},allow_unicode=True),encoding="utf-8")


## Huấn luyện hoặc nạp trọng số gán nhãn sơ bộ
Tắt mọi phép lật, ghép ảnh và biến đổi hình học; dùng các biến thể camera đã tạo ở trên. Chế độ thật bắt đầu từ trọng số COCO và lưu trọng số tốt nhất. Chế độ giả khởi tạo từ mô tả kiến trúc YAML, không dùng chất lượng dự đoán để kết luận.

`MODE="prelabel"` nạp `PRELABEL_WEIGHTS` và kiểm tra tên lớp khớp dataset. Phần đánh giá, xuất ONNX bên dưới vẫn có thể chạy để đối chiếu trọng số này với phiên bản dữ liệu đã chọn.


In [ ]:
if MODE=="prelabel" and not FAKE_DATA:
    weights=Path(PRELABEL_WEIGHTS).expanduser().resolve()
    if not weights.is_file() or ROOT not in weights.parents:
        raise ValueError("PRELABEL_WEIGHTS phải là file bên trong Vong3")
    detector=YOLO(str(weights))
else:
    # Đặt thư mục hiện tại trong Vong3 để trọng số tải xuống cũng nằm ở đây.
    previous=Path.cwd()
    try:
        os.chdir(RUN)
        detector=YOLO(MODEL+".yaml" if FAKE_DATA else MODEL+".pt")
        detector.train(data=str(DATA_YAML.resolve()),imgsz=SIZE,epochs=1 if FAKE_DATA else EPOCHS,
                       batch=2 if FAKE_DATA else BATCH,device=DEVICE,workers=0,seed=SEED,
                       deterministic=True,pretrained=not FAKE_DATA,amp=False,
                       fliplr=0.0,flipud=0.0,mosaic=0.0,mixup=0.0,copy_paste=0.0,
                       hsv_h=0.0,hsv_s=0.0,hsv_v=0.0,degrees=0.0,translate=0.0,scale=0.0,
                       shear=0.0,perspective=0.0,plots=False,save=True,cache=False,
                       project=str(RUN/"training"),name="fit",exist_ok=False)
        weights=Path(detector.trainer.best)
        shutil.copy2(weights,RUN/"yolo.pt")
    finally:
        os.chdir(previous)
    weights=RUN/"yolo.pt"
    detector=YOLO(str(weights))
assert [detector.names[i] for i in range(NC)]==CLASS_NAMES, "Thứ tự lớp không khớp"


## Ma trận nhầm lẫn, recall và ảnh lỗi
Recall (tỉ lệ tìm đúng trên tổng số vật thật) và precision (tỉ lệ đúng trên các dự đoán) dùng ghép khung một-một ở IoU ≥ 0,5. Ma trận có hàng nhãn thật, cột dự đoán; hàng/cột cuối là nền. Khung trùng vị trí nhưng sai lớp được ghi vào ô ngoài đường chéo, đồng thời tính một lần bỏ sót và một lần báo nhầm. Quét ngưỡng conf (độ tin cậy) và năm mức độ sáng. Ảnh lỗi kèm phiên, lượt, mục tiêu, pass (lần đi qua vị trí).


In [ ]:
def xywh_to_xyxy(a):
    a=np.asarray(a,dtype=float).reshape(-1,4)
    return np.concatenate([a[:,:2]-a[:,2:]/2,a[:,:2]+a[:,2:]/2],axis=1)

def pair_iou(a,b):
    left=np.maximum(a[:,None,:2],b[None,:,:2]); right=np.minimum(a[:,None,2:],b[None,:,2:])
    inter=np.maximum(right-left,0).prod(2)
    aa=np.maximum(a[:,2:]-a[:,:2],0).prod(1); bb=np.maximum(b[:,2:]-b[:,:2],0).prod(1)
    return inter/(aa[:,None]+bb[None,:]-inter+1e-9)

def confusion_one(gt,pred):
    # gt: lớp,x1,y1,x2,y2; pred: lớp,conf,x1,y1,x2,y2 chuẩn hoá.
    mat=np.zeros((NC+1,NC+1),dtype=int)
    used_g=set(); used_p=set()
    if len(gt) and len(pred):
        ious=pair_iou(gt[:,1:],pred[:,2:])
        candidates=np.argwhere(ious>=MATCH_IOU)
        candidates=sorted(candidates,key=lambda z:-ious[z[0],z[1]])
        for g,p in candidates:
            g=int(g); p=int(p)
            if g in used_g or p in used_p: continue
            used_g.add(g); used_p.add(p); mat[int(gt[g,0]),int(pred[p,0])]+=1
    for g in set(range(len(gt)))-used_g: mat[int(gt[g,0]),NC]+=1
    for p in set(range(len(pred)))-used_p: mat[NC,int(pred[p,0])]+=1
    return mat

PRED={}
for gain in BRIGHTNESS:
    for r in val:
        im=cv2.imread(str(r["image"])); im=np.clip(im.astype(float)*gain,0,255).astype(np.uint8)
        res=detector.predict(im,imgsz=SIZE,conf=min(min(CONF_THRESHOLDS),EVAL_CONF),iou=NMS_IOU,
                             device=DEVICE,verbose=False,rect=False)[0]
        PRED[(gain,r["stem"])]=np.column_stack([res.boxes.cls.cpu().numpy(),res.boxes.conf.cpu().numpy(),
                                                res.boxes.xyxyn.cpu().numpy()]).reshape(-1,6)

def evaluate(gain,conf,with_errors=False):
    total=np.zeros((NC+1,NC+1),int); errors=[]
    for r in val:
        labels=GT[r["stem"]]
        gt=np.column_stack([labels[:,0],xywh_to_xyxy(labels[:,1:])]).reshape(-1,5)
        p=PRED[(gain,r["stem"])]; p=p[p[:,1]>=conf]
        mat=confusion_one(gt,p); total+=mat
        wrong=int(mat.sum()-np.trace(mat))
        if with_errors and wrong:
            errors.append({k:r[k] for k in ("stem","session","luot","target","pass_id")}|{"errors":wrong,"brightness":gain,"conf":conf})
    metrics=[]
    for c,n in enumerate(CLASS_NAMES):
        tp=int(total[c,c]); actual=int(total[c,:].sum()); predicted=int(total[:,c].sum())
        metrics.append(dict(brightness=gain,conf=conf,cls=n,tp=tp,fp=predicted-tp,fn=actual-tp,
                            recall=tp/actual if actual else None,precision=tp/predicted if predicted else None))
    return metrics,total,errors
all_metrics=[]; all_errors=[]
for gain in BRIGHTNESS:
    for conf in sorted(set(CONF_THRESHOLDS+[EVAL_CONF])):
        rows,mat,errors=evaluate(gain,conf,with_errors=conf==EVAL_CONF)
        all_metrics+=rows; all_errors+=errors
        if gain==1.0 and conf==EVAL_CONF:
            CONFUSION=mat
write_csv(RUN/"yolo_metrics.csv",all_metrics,["brightness","conf","cls","tp","fp","fn","recall","precision"])
write_csv(RUN/"error_images.csv",sorted(all_errors,key=lambda r:-r["errors"]),["stem","session","luot","target","pass_id","errors","brightness","conf"])
with (RUN/"confusion_matrix.csv").open("w",encoding="utf-8-sig",newline="") as f:
    writer=csv.writer(f); writer.writerow(["nhan_that_du_doan"]+CLASS_NAMES+["nen"])
    for n,row in zip(CLASS_NAMES+["nen"],CONFUSION): writer.writerow([n]+row.tolist())
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(max(5,NC),max(4,NC)))
ax.imshow(CONFUSION,cmap="Blues")
ax.set(xticks=range(NC+1),yticks=range(NC+1),xticklabels=CLASS_NAMES+["nền"],yticklabels=CLASS_NAMES+["nền"],
       xlabel="Dự đoán",ylabel="Nhãn thật")
plt.setp(ax.get_xticklabels(),rotation=45,ha="right")
for (i,j),v in np.ndenumerate(CONFUSION): ax.text(j,i,str(v),ha="center",va="center")
fig.tight_layout(); fig.savefig(RUN/"confusion_matrix.png"); plt.close(fig)
print("Chỉ số từng lớp, độ sáng 1.0, conf",EVAL_CONF)
print(*[r for r in all_metrics if r["brightness"]==1.0 and r["conf"]==EVAL_CONF],sep="\n")


## Gán nhãn sơ bộ trên ảnh đang chờ của Roboflow
Đặt `MODE="prelabel"` và đường dẫn trọng số đã học. Chỉ tìm ảnh chưa vào Dataset (tập đã duyệt), ghi dự đoán trên đúng mã ảnh, không ghi đè chú thích đã có, không tự đưa ảnh vào Dataset. Ảnh không tìm thấy vật được ghi nhật ký để người kiểm tra. Chạy lại sẽ bỏ mã ảnh đã ghi thành công; lỗi lưu sẽ dừng để tránh báo thành công giả. Chế độ dữ liệu giả chạy thử phần tạo nhãn và nhật ký, không gọi Roboflow.


In [ ]:
if MODE=="prelabel" or FAKE_DATA:
    import requests
    PRE=RUN/"prelabel"; PRE.mkdir(exist_ok=True)
    journal=ROOT/"yolo"/"prelabel_done.json"
    done=json.loads(journal.read_text(encoding="utf-8")) if journal.exists() and not FAKE_DATA else []
    if FAKE_DATA:
        pending=[{"id":r["stem"],"name":r["image"].name,"local_image":r["image"]} for r in val[:2]]
    else:
        project=rf_project(); pending=[]; offset=0
        while True:
            page=project.search(in_dataset=False,offset=offset,limit=250,fields=["id","name"])
            if not isinstance(page,list): raise RuntimeError("Định dạng search Roboflow thay đổi; cần kiểm tra SDK")
            pending.extend(page)
            if len(page)<250: break
            offset+=250
        done_set=set(done)
        prefix="{}/{}:".format(WORKSPACE or "default",PROJECT)
        pending=[p for p in pending if prefix+str(p["id"]) not in done_set]
    if PRELABEL_LIMIT is not None: pending=pending[:PRELABEL_LIMIT]
    logs=[]
    for item in pending:
        if FAKE_DATA:
            im=cv2.imread(str(item["local_image"]))
        else:
            data=project.image(item["id"])
            urls=data.get("urls") or {}; url=urls.get("original") or urls.get("annotation") or data.get("image_url")
            if not url: raise RuntimeError("Roboflow không trả URL ảnh")
            response=requests.get(url,timeout=(10,30)); response.raise_for_status()
            im=cv2.imdecode(np.frombuffer(response.content,np.uint8),cv2.IMREAD_COLOR)
        if im is None: raise RuntimeError("Ảnh gán sơ bộ không đọc được")
        result=detector.predict(im,imgsz=SIZE,conf=PRELABEL_CONF,iou=NMS_IOU,device=DEVICE,verbose=False,rect=False)[0]
        # Mã băm tránh ghi tên do máy chủ trả về thành đường dẫn không hợp lệ.
        safe=hashlib.sha256(str(item["id"]).encode()).hexdigest()[:20]
        label=PRE/(safe+".txt")
        rows=[]
        for c,box in zip(result.boxes.cls.cpu().numpy(),result.boxes.xywhn.cpu().numpy()):
            rows.append("{} {}".format(int(c)," ".join("{:.6f}".format(float(v)) for v in box)))
        label.write_text("\n".join(rows)+("\n" if rows else ""),encoding="utf-8")
        status="khong_thay_vat"
        if rows:
            if FAKE_DATA:
                status="gia_lap_luu_nhan"
            else:
                project.save_annotation(annotation_path=str(label),annotation_labelmap=dict(enumerate(CLASS_NAMES)),
                                        image_id=item["id"],is_prediction=True,annotation_overwrite=False,add_to_dataset=False)
                status="da_gui_du_doan_can_duyet"
        logs.append(dict(image_id=item["id"],name=item["name"],boxes=len(rows),status=status))
        if not FAKE_DATA and rows:
            done.append(prefix+str(item["id"])); write_json(journal,sorted(set(done)))
        write_csv(PRE/"prelabel_log.csv",logs,["image_id","name","boxes","status"])
    print("Ảnh xử lý:",len(logs),"| Cần mở Roboflow để duyệt nhãn.")
else:
    print("Chế độ train: bỏ qua ghi nhãn sơ bộ.")


## Xuất ONNX cố định và so với PyTorch
Đầu vào RGB, NCHW (lô ảnh, kênh, cao, rộng), float32 /255; letterbox (giữ tỉ lệ, đệm 114, căn giữa) ngoài mô hình. ONNX xuất opset 12, FP32, không NMS; đầu ra `[1,4+nc,N]` chứa `cx,cy,w,h` theo pixel ảnh đầu vào và xác suất từng lớp. Suy luận trên xe cần NMS theo lớp.

So sánh ít nhất 20 ảnh, sai số điểm lớp < 0,001; sai số toạ độ tối đa < 0,01 pixel. Lưu lại cùng tensor (mảng nhiều chiều) đầu vào và đầu ra PyTorch để kiểm tra TensorRT sau. Ước lượng tốc độ từ số phép tính không phải số đo trên Nano.


In [ ]:
from ultralytics.data.augment import LetterBox
from ultralytics.utils.torch_utils import get_flops
# Dùng cùng cách đếm của Ultralytics; get_flops đã trả số phép tính, không nhân 2 lần nữa.
PARAMS=int(sum(p.numel() for p in detector.model.parameters()))
GFLOPS=float(get_flops(detector.model,imgsz=SIZE))
if GFLOPS<=0: raise RuntimeError("Không đếm được GFLOPs")
print("Tham số:",PARAMS,"| GFLOPs:",GFLOPS,"| Nano ước lượng ms:",6*GFLOPS)
exported=Path(detector.export(format="onnx",imgsz=SIZE,batch=1,opset=12,simplify=True,
                              dynamic=False,half=False,nms=False,device="cpu"))
onnx_path=RUN/"yolo.onnx"
if exported.resolve()!=onnx_path.resolve(): shutil.copy2(exported,onnx_path)
if weights.resolve()!=(RUN/"yolo.pt").resolve(): shutil.copy2(weights,RUN/"yolo.pt")
graph=onnx.load(str(onnx_path)); onnx.checker.check_model(graph)
assert next(v.version for v in graph.opset_import if v.domain in ("","ai.onnx"))==12
assert [d.dim_value for d in graph.graph.input[0].type.tensor_type.shape.dim]==[1,3,SIZE,SIZE]
session=ort.InferenceSession(str(onnx_path),providers=["CPUExecutionProvider"])
pt=YOLO(str(RUN/"yolo.pt")).model.cpu().float().eval()
letterbox=LetterBox(new_shape=(SIZE,SIZE),auto=False,scale_fill=False,scaleup=True,stride=32)
check_records=val+train
if len(check_records)<20: raise ValueError("Cần ít nhất 20 ảnh để đối chiếu ONNX")
checks=[]; check_dir=RUN/"onnx_reference"; check_dir.mkdir()
for i,r in enumerate(check_records[:max(20,len(val))]):
    im=letterbox(image=cv2.imread(str(r["image"])))
    x=np.ascontiguousarray(im[:,:,::-1].transpose(2,0,1)[None],dtype=np.float32)/255
    with torch.no_grad():
        a=pt(torch.from_numpy(x)); a=a[0] if isinstance(a,(tuple,list)) else a
        a=a.numpy()
    b=session.run(None,{session.get_inputs()[0].name:x})[0]
    assert a.shape==b.shape and b.shape[1]==4+NC
    score_error=float(np.max(np.abs(a[:,4:]-b[:,4:])))
    box_error=float(np.max(np.abs(a[:,:4]-b[:,:4])))
    assert score_error<1e-3 and box_error<1e-2, "Kết quả ONNX lệch PyTorch"
    np.savez_compressed(check_dir/("{:03d}.npz".format(i)),images=x,output=a)
    checks.append(dict(stem=r["stem"],max_score_error=score_error,max_box_error_px=box_error))
write_csv(RUN/"onnx_check.csv",checks,["stem","max_score_error","max_box_error_px"])
import ultralytics
info={"task":"detection","architecture":MODEL,"class_names":CLASS_NAMES,"num_classes":NC,
      "input_name":session.get_inputs()[0].name,"input_shape":[1,3,SIZE,SIZE],"input_hw":[SIZE,SIZE],
      "input_dtype":"float32","color_order":"RGB","layout":"NCHW","normalization":{"scale":1/255,"mean":[0,0,0],"std":[1,1,1]},
      "preprocessing_in_model":False,"resize":"letterbox","letterbox":{"auto":False,"center":True,"padding_value":114,"scaleup":True,"stride":32},
      "output_name":session.get_outputs()[0].name,"output_shape":list(b.shape),"output_type":"cxcywh_pixels_and_class_probabilities",
      "box_format":"cxcywh","box_units":"input_pixels","scores_are_probabilities":True,"objectness":False,
      "nms_in_model":False,"nms_class_aware":True,"conf_threshold":EVAL_CONF,"nms_iou":NMS_IOU,
      "opset":12,"dynamic":False,"parameters":PARAMS,"gflops":GFLOPS,"flops_convention":"2 FLOPs per MAC",
      "nano_estimated_ms":6*GFLOPS,"latency_measured":False,"seed":SEED,"fake_data":FAKE_DATA,
      "project":PROJECT,"version":VERSION,"ultralytics_version":ultralytics.__version__,"torch_version":torch.__version__,
      "onnxruntime_version":ort.__version__,"onnx_check_images":len(checks),
      "onnx_max_score_error":max(c["max_score_error"] for c in checks),"onnx_max_box_error_px":max(c["max_box_error_px"] for c in checks)}
write_json(RUN/"yolo_info.json",info)
print("Đã lưu:",RUN)
print("Trên Jetson: /usr/src/tensorrt/bin/trtexec --onnx=yolo.onnx --saveEngine=yolo_fp16.engine --fp16 --workspace=1024")
